# 01 - Build the EDA subset

**Goal.** Turn DICOM slices into clean 3D volumes in Hounsfield units, with the slice
labels aligned to the volume's z-axis, and store them in shards (on Drive, or in the
Kaggle output folder). On Colab the slices are streamed from the public zip; on Kaggle
they are read from the mounted dataset, so this notebook is optional there: the EDA
notebook builds any missing volume itself.

Glossary for this notebook:

- **Slice**: one 2D frame of the 3D volume (one DICOM file).
- **Axial / coronal / sagittal**: the three viewing planes. Think of the body as a loaf of
  bread standing upright: *axial* slices are the normal bread slices seen from above,
  *coronal* cuts it front-to-back (a view from the front), *sagittal* cuts it left-to-right
  (a view from the side).
- **HU**: Hounsfield units, the calibrated density scale (air -1000, water 0, bone about +1000).
- **Series assembly**: each slice file is read, the slices are sorted by their physical
  position, the scanner's intensity calibration is applied, and the 3D geometry (voxel
  size, origin, orientation) is built from the headers.
- **z-axis**: the head-to-feet axis. In our stored volumes index 0 is the top of the scan.

In [ ]:
import os, sys, subprocess

# Python 3.13 removed 'imp' but IPython's autoreload still imports it
if sys.version_info >= (3, 13) and "imp" not in sys.modules:
    import importlib, types
    imp_mock = types.ModuleType("imp"); imp_mock.reload = importlib.reload; sys.modules["imp"] = imp_mock

BRANCH = "main"
REPO = "https://github.com/SattamAltwaim/rsna-pe-ct.git"
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = os.path.exists("/content") and not ON_KAGGLE
DEST = "/kaggle/working/rsna-pe-ct" if ON_KAGGLE else "/content/rsna-pe-ct"

if ON_COLAB or ON_KAGGLE:                            # hosted: clone or fast-forward, install
    if not os.path.exists(DEST):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO, DEST], check=True)
    else:
        subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", DEST], check=True)
    if DEST not in sys.path:
        sys.path.insert(0, DEST)
    if ON_COLAB:
        from google.colab import drive; drive.mount("/content/drive")
else:                                                # local fallback (Mac)
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "pe_ct")) and parent not in sys.path:
        sys.path.insert(0, parent)

%load_ext autoreload
%autoreload 2
import pe_ct
print("pe_ct", pe_ct.__version__, "| commit:",
      subprocess.run(["git", "-C", DEST if os.path.exists(DEST) else "..", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip())

## Configuration

In [ ]:
import os
from pe_ct.config import default_config
from pe_ct import viz

NOTEBOOK = "01_download_subset"
cfg = default_config()                       # Kaggle: mounted dataset + /kaggle/working; Colab: S3 + Drive; local: ./data
SMOKE = os.environ.get("PE_CT_SMOKE") == "1" # tiny sizes for a quick end-to-end check
if SMOKE:
    cfg.n_eda, cfg.n_dev, cfg.n_test_sample, cfg.shard_size = 4, 60, 20, 10

cfg.ensure_dirs()
if cfg.source == "kaggle":                   # outputs of earlier notebooks attached as inputs
    from pe_ct import colab
    from pe_ct.config import describe_kaggle_inputs
    print("competition data:", cfg.kaggle_input or "NOT FOUND - attach it via Add Input -> Competitions")
    print("/kaggle/input contains:"); print(describe_kaggle_inputs())
    print("linked earlier outputs:", colab.link_kaggle_inputs(cfg))
viz.use_notebook_style()
FIG = cfg.figures_dir
print(cfg)

In [ ]:
from pe_ct import colab
report = colab.session_report(cfg)

## Load the split table and select the EDA subset

We also build two lookup objects: one that locates a study's files (byte offsets in the
zip on the S3 source; nothing needed on Kaggle), and one that maps a study to its
per-slice labels.

In [ ]:
import pandas as pd
from pe_ct import labels, pipeline

studies, split_table = pipeline.ensure_labels_and_splits(cfg)
eda_uids = split_table.loc[split_table.in_eda, "study_uid"].tolist()
print(len(eda_uids), "EDA studies |", int(split_table.loc[split_table.in_eda, "y"].sum()), "PE positive")

In [ ]:
locator = pipeline.make_locator(cfg)          # zip offsets (S3) or None (Kaggle)
slice_index = labels.SliceLabelIndex(pipeline.load_train(cfg))
print("source:", cfg.source)

## One study through the pipeline

Fetch, decode, orient, align labels. The printout checks the three things that silently
break everything if wrong: the shape, the voxel spacing, and the HU range (air must sit
near -1000; contrast-filled vessels and bone must reach several hundred).

In [ ]:
from pe_ct import volume

uid = eda_uids[0]
image, meta, timings = pipeline.process_study(cfg, locator, slice_index, uid, studies.set_index("study_uid").loc[uid], cfg.work / "dicom")
vol = volume.image_to_hu_array(image)
print("study", uid, "| shape (z, y, x)", vol.shape, "| spacing (x, y, z) mm", [round(s, 3) for s in meta["spacing_xyz_mm"]])
print("timings", timings, "| scanner", meta["manufacturer"], "| slice thickness", meta["slice_thickness_mm"], "mm")
print("HU sanity", meta["hu_sanity"])
print("positive slices:", labels.positive_runs(meta["slice_labels"]), "of", len(meta["slice_labels"]))

### Three orthogonal views of one study

**What you're looking at.** Left: an axial slice (seen from the feet, the standard radiology view: the patient's front is at the top, their left is on the right of the screen). Middle: a coronal cut (seen from the front). Right: a sagittal cut (seen from the side). Orange lines show where the other two planes cut. The thin strip on the far right marks PE-positive slices in red along the z-axis. Vessel window.

**Why it matters.** This is the visual proof that the volume was assembled correctly: slices in the right order, the head at the top, no left-right mirroring, and voxel spacing applied so the body is not squashed.

**What to look for.** A chest that looks like a chest: lungs dark, heart and large vessels bright, spine at the back. The red band should fall at the level of the heart, where the lung arteries are.

In [ ]:
z = labels.representative_positive_slice(meta["slice_labels"]) or vol.shape[0] // 2
fig = viz.orthoviews(vol, meta, window="vessel", z=z, labels=meta["slice_labels"], title=f"study {uid}")
viz.save_fig(fig, FIG, NOTEBOOK, "orthoviews_one_study");

## Build the EDA volumes (resumable)

Each study is fetched (parallel range requests on the S3 source; read in place on Kaggle),
decoded into a volume and appended to the current shard; downloaded DICOM files are
deleted afterwards. A shard is moved to the output folder and gets its `.done.json`
marker only when complete, so a disconnected session loses at most one partial shard.
Studies that fail are logged with the reason and skipped on re-runs.

In [ ]:
from tqdm.auto import tqdm
from pe_ct import storage

failure_log = storage.FailureLog(cfg.failures_csv(NOTEBOOK))
stats = pipeline.run_volume_download(cfg, studies, eda_uids, locator, slice_index, failure_log, progress=tqdm)

## Summary

The measured seconds per study is the number that sets the budget for the embedding
notebook, which streams many more studies through the same fetch-and-decode path.

In [ ]:
import numpy as np

done_uids = storage.done_uids(cfg.volumes_eda_dir, "tar")
secs = np.array(stats["seconds"]) if stats["seconds"] else np.array([np.nan])
print(f"done this run: {stats['done']} | failed: {stats['failed']} | stored in total: {len(done_uids)} of {len(eda_uids)}")
print(f"seconds per study: median {np.nanmedian(secs):.1f} | mean {np.nanmean(secs):.1f}")
print(f"GB written this run: {stats['bytes'] / 2**30:.2f} | output free: {colab.free_space_gb(cfg.root)} GB")
failure_log.df.tail()

## Local copy of the shards

Reading many files from Drive is slow, so the finished shards are copied to local disk
and extracted once. The EDA notebook does the same at its start.

In [ ]:
fresh = storage.prepare_local(cfg.volumes_eda_dir, cfg.work / "volumes_eda", progress=tqdm)
store = storage.VolumeStore(cfg.work / "volumes_eda" / "cache")
print(len(fresh), "shards extracted now |", len(store.uids()), "volumes available locally")

### Coronal views of random volumes

**What you're looking at.** A grid of coronal (front) views through the middle of randomly chosen stored volumes, vessel window, head at the top. The caption gives the study id, the label and the scanner when known.

**Why it matters.** A quick visual check across different scanners: every panel must look like a chest. One wrong-way-up or mirrored volume here would mean the orientation code fails on some scanner's conventions.

**What to look for.** Head at the top, lungs as two dark lobes, the bright heart between them, the spine running down the middle. Aspect ratios should look natural (slices are thicker than in-plane pixels, which the plot corrects for).

In [ ]:
rng = np.random.default_rng(cfg.seed)
sample = rng.choice(store.uids(), size=min(5, len(store.uids())), replace=False)
images, titles, aspects = [], [], []
for u in sample:
    v, m = store.load_volume(u)
    images.append(volume.coronal_slab(v)); aspects.append(volume.aspect_coronal(m))
    titles.append(f"{u}  y={m['y']}  {m.get('scanner', m['manufacturer'])[:14]}")
fig = viz.image_grid(images, titles, ncols=5, window="vessel", aspects=aspects)
viz.save_fig(fig, FIG, NOTEBOOK, "coronal_random_volumes");

## What we learned

    - (fill in after running) seconds per study and where the time goes (network vs decode).
- (fill in after running) how many studies failed and why (compressed pixel data, missing slices, odd geometry).
- (fill in after running) whether any scanner needed special handling for orientation.
- Volumes are stored head-first in LPI orientation, so slice 0 is the top of the scan in every notebook.